# 04 — Optimized Voxel Experimental Validation / Dataset Update

최적 Voxel 제작 후 **실제 descriptor와 compression curve**를 기존 registry에 연결합니다. 예측–실험 오차를 저장하고, 다음 retraining cycle에서 자동으로 새 데이터가 포함되게 합니다.


In [ ]:
from pathlib import Path
import sys,joblib
PROJECT_POINTER=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Voxel generation\.ai_voxel_ml_project.json")
CODE_DIR=Path.cwd()/"Code" if (Path.cwd()/"Code").exists() else Path.cwd();sys.path.insert(0,str(CODE_DIR)) if str(CODE_DIR) not in sys.path else None
from voxel_ml_common import load_contract
_contract0=load_contract(PROJECT_POINTER)
# DatasetFactory를 inverse_design/active_sampling mode로 실행한 직후라면 최신 Result snapshot을 자동 사용합니다.
VALIDATION_DESCRIPTOR_CSV=Path(_contract0.get('current_run_snapshot_csv',_contract0['all_structures_csv']))
# 실제 압축시험 long-format curve 파일 위치만 사용자가 지정합니다.
VALIDATION_COMPRESSION_LONG=Path(r"C:\Users\Administrator\Desktop\Minkyeom\AI-Voxel\Compression test\inverse_validation_curves.csv")
PROMOTE_VALIDATION_TO_MASTER=True


In [ ]:
import numpy as np,pandas as pd,joblib,matplotlib.pyplot as plt
from voxel_ml_common import load_contract,atomic_csv,atomic_json
from compression_curve_model import load_canonical_long,build_curve_dataset,predict_curve,extract_curve_properties
c=load_contract(PROJECT_POINTER);mr=Path(c['model_root']);val_dir=Path(c.get('validation_root',Path(c['data_root'])/'validation_cycles'));val_dir.mkdir(parents=True,exist_ok=True)
if not VALIDATION_DESCRIPTOR_CSV.exists(): raise FileNotFoundError(VALIDATION_DESCRIPTOR_CSV)
if not VALIDATION_COMPRESSION_LONG.exists(): raise FileNotFoundError(VALIDATION_COMPRESSION_LONG)
v=pd.read_csv(VALIDATION_DESCRIPTOR_CSV);gen=joblib.load(mr/'01_gen2desc'/'gen2desc_bundle.joblib');curve=joblib.load(mr/'02_desc2curve'/'desc2curve_bundle.joblib')
curves=load_canonical_long(VALIDATION_COMPRESSION_LONG);tab,Y,M,grid=build_curve_dataset(curves,v,gen,grid_points=len(curve['strain_grid']),grid_max=float(curve['strain_grid'][-1]),prefer_final=True)
Z=tab[[cc for cc in tab if cc.startswith('desc_latent_')]].to_numpy(float);pr=predict_curve(curve,Z);P=pr['curve_mean'];rows=[]
for i in range(len(tab)):
    mask=M[i]>0.5;rmse=float(np.sqrt(np.mean((Y[i,mask]-P[i,mask])**2))) if mask.any() else np.nan
    r2=1-float(np.sum((Y[i,mask]-P[i,mask])**2))/max(float(np.sum((Y[i,mask]-np.mean(Y[i,mask]))**2)),1e-12) if mask.sum()>2 else np.nan
    row={'specimen_id':tab.iloc[i].specimen_id,'design_id':tab.iloc[i].design_id,'curve_rmse_MPa':rmse,'curve_r2':r2}
    truep=extract_curve_properties(grid,Y[i],M[i]);predp=extract_curve_properties(grid,P[i],M[i])
    for k in sorted(set(truep)&set(predp)):
        if np.isscalar(truep[k]) and np.isscalar(predp[k]): row[k+'__measured']=truep[k];row[k+'__predicted']=predp[k];row[k+'__error']=predp[k]-truep[k]
    rows.append(row)
res=pd.DataFrame(rows);atomic_csv(res,val_dir/'latest_prediction_validation.csv');print('Validation mean curve RMSE:',res.curve_rmse_MPa.mean(),'MPa');display(res)
fig,ax=plt.subplots(figsize=(8,5))
for i in range(min(6,len(tab))):
    ax.plot(grid,Y[i],lw=1.5,label=f'{tab.iloc[i].specimen_id} measured');ax.plot(grid,P[i],'--',lw=1.2)
ax.set_xlabel('Strain');ax.set_ylabel('Stress (MPa)');ax.set_title('Optimized-design validation: solid=measured, dashed=predicted');ax.grid(alpha=.2);fig.tight_layout();fig.savefig(val_dir/'latest_curve_validation.png',dpi=220);plt.show()


In [ ]:
import pandas as pd,json
from voxel_ml_common import load_contract,atomic_csv,atomic_json
c=load_contract(PROJECT_POINTER);root=Path(c['data_root']);val_dir=Path(c.get('validation_root',root/'validation_cycles'));val_dir.mkdir(parents=True,exist_ok=True)  # data_root is the persistent cumulative project dir (matches cell 2's val_dir), not the current run's ephemeral ml_root
if not VALIDATION_DESCRIPTOR_CSV.exists():raise FileNotFoundError(VALIDATION_DESCRIPTOR_CSV)
v=pd.read_csv(VALIDATION_DESCRIPTOR_CSV);master=pd.read_csv(c['all_structures_csv']);combined=pd.concat([master,v],ignore_index=True,sort=False)
# Geometry/fidelity-aware duplicate removal; latest validation row wins.
keys=[k for k in ['sample_id','meta__fidelity','meta__geometry_id'] if k in combined]
if keys:combined=combined.drop_duplicates(keys,keep='last')
atomic_csv(combined,root/'all_structures_with_validation.csv')
cycle={'validation_descriptor_source':str(VALIDATION_DESCRIPTOR_CSV),'validation_curve_source':str(VALIDATION_COMPRESSION_LONG),'rows_added':len(v),'combined_rows':len(combined)};atomic_json(val_dir/'latest_validation_cycle.json',cycle)
print(json.dumps(cycle,indent=2,ensure_ascii=False));display(v.head())

# Append validation compression curves to persistent central master.
if VALIDATION_COMPRESSION_LONG.exists():
    newc=pd.read_csv(VALIDATION_COMPRESSION_LONG); master_curve=Path(c['compression_curve_master'])
    oldc=pd.read_csv(master_curve) if master_curve.exists() else pd.DataFrame(); cc=pd.concat([oldc,newc],ignore_index=True,sort=False)
    key=[x for x in ['specimen_id','replicate_id','strain'] if x in cc]
    if key: cc=cc.drop_duplicates(key,keep='last')
    atomic_csv(cc,master_curve); cycle['compression_rows_master']=len(cc)
if PROMOTE_VALIDATION_TO_MASTER:
    global_path=Path(c['all_structures_csv']); atomic_csv(combined,global_path); cycle['promoted_to_master']=True
atomic_json(val_dir/'latest_validation_cycle.json',cycle)
print('Retrain Notebook 01 and 02 after promotion.')


### Retraining
검증 데이터를 정식 master로 승격하려면 `project_contract.json`의 `all_structures_csv`를 `all_structures_with_validation.csv`로 바꾸거나, 새 DatasetFactory run을 project pointer로 지정한 뒤 Notebook 01→02를 재실행합니다. 압축곡선도 기존 training curve 파일에 append한 후 Model 2를 재학습합니다.
